**Module** is an abstract class which defines fundamental methods necessary for a training a neural network. You do not need to change anything here, just read the comments.

In [1]:
import numpy as np

In [2]:
class Module(object):
    """
    Basically, you can think of a module as of a something (black box)
    which can process `input` data and produce `ouput` data.
    This is like applying a function which is called `forward`:

        output = module.forward(input)

    The module should be able to perform a backward pass: to differentiate the `forward` function.
    More, it should be able to differentiate it if is a part of chain (chain rule).
    The latter implies there is a gradient from previous step of a chain rule.

        gradInput = module.backward(input, gradOutput)
    """
    def __init__ (self):
        self.output = None
        self.gradInput = None
        self.training = True

    def forward(self, input):
        """
        Takes an input object, and computes the corresponding output of the module.
        """
        return self.updateOutput(input)

    def backward(self,input, gradOutput):
        """
        Performs a backpropagation step through the module, with respect to the given input.

        This includes
         - computing a gradient w.r.t. `input` (is needed for further backprop),
         - computing a gradient w.r.t. parameters (to update parameters while optimizing).
        """
        self.updateGradInput(input, gradOutput)
        self.accGradParameters(input, gradOutput)
        return self.gradInput


    def updateOutput(self, input):
        """
        Computes the output using the current parameter set of the class and input.
        This function returns the result which is stored in the `output` field.

        Make sure to both store the data in `output` field and return it.
        """

        # The easiest case:

        # self.output = input
        # return self.output

        pass

    def updateGradInput(self, input, gradOutput):
        """
        Computing the gradient of the module with respect to its own input.
        This is returned in `gradInput`. Also, the `gradInput` state variable is updated accordingly.

        The shape of `gradInput` is always the same as the shape of `input`.

        Make sure to both store the gradients in `gradInput` field and return it.
        """

        # The easiest case:

        # self.gradInput = gradOutput
        # return self.gradInput

        pass

    def accGradParameters(self, input, gradOutput):
        """
        Computing the gradient of the module with respect to its own parameters.
        No need to override if module has no parameters (e.g. ReLU).
        """
        pass

    def zeroGradParameters(self):
        """
        Zeroes `gradParams` variable if the module has params.
        """
        pass

    def getParameters(self):
        """
        Returns a list with its parameters.
        If the module does not have parameters return empty list.
        """
        return []

    def getGradParameters(self):
        """
        Returns a list with gradients with respect to its parameters.
        If the module does not have parameters return empty list.
        """
        return []

    def train(self):
        """
        Sets training mode for the module.
        Training and testing behaviour differs for Dropout, BatchNorm.
        """
        self.training = True

    def evaluate(self):
        """
        Sets evaluation mode for the module.
        Training and testing behaviour differs for Dropout, BatchNorm.
        """
        self.training = False

    def __repr__(self):
        """
        Pretty printing. Should be overrided in every module if you want
        to have readable description.
        """
        return "Module"

# Sequential container

**Define** a forward and backward pass procedures.

In [1]:
class Sequential(Module):
    def __init__ (self):
        super(Sequential, self).__init__()
        self.modules = []

    def add(self, module):
        self.modules.append(module)

    def updateOutput(self, input):
        self.forward_inputs = [input]
        x = input

        for module in self.modules:
            x = module.forward(x)
            self.forward_inputs.append(x)

        self.output = x
        return self.output

    def backward(self, input, gradOutput):
        grad = gradOutput

        for i in range(len(self.modules) - 1, -1, -1):
            module = self.modules[i]
            layer_input = self.forward_inputs[i]
            grad = module.backward(layer_input, grad)

        self.gradInput = grad
        return self.gradInput

    def zeroGradParameters(self):
        for module in self.modules:
            module.zeroGradParameters()

    def getParameters(self):
        return [x.getParameters() for x in self.modules]

    def getGradParameters(self):
        return [x.getGradParameters() for x in self.modules]

    def __repr__(self):
        string = "".join([str(x) + '\n' for x in self.modules])
        return string

    def __getitem__(self, x):
        return self.modules.__getitem__(x)

    def train(self):
        self.training = True
        for module in self.modules:
            module.train()

    def evaluate(self):
        self.training = False
        for module in self.modules:
            module.evaluate()

NameError: name 'Module' is not defined

# Layers

## 1 (0.2). Linear transform layer
Also known as dense layer, fully-connected layer, FC-layer, InnerProductLayer (in caffe), affine transform
- input:   **`batch_size x n_feats1`**
- output: **`batch_size x n_feats2`**

In [4]:
class Linear(Module):
    """
    A module which applies a linear transformation
    A common name is fully-connected layer, InnerProductLayer in caffe.

    The module should work with 2D input of shape (n_samples, n_feature).
    """
    def __init__(self, n_in, n_out):
        super(Linear, self).__init__()

        # This is a nice initialization
        stdv = 1./np.sqrt(n_in)
        self.W = np.random.uniform(-stdv, stdv, size = (n_out, n_in))
        self.b = np.random.uniform(-stdv, stdv, size = n_out)

        self.gradW = np.zeros_like(self.W)
        self.gradb = np.zeros_like(self.b)

    def updateOutput(self, input):
        # Your code goes here. ################################################
        self.output = input @ self.W.T + self.b
        return self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        # self.gradInput = ...
        self.gradInput = gradOutput @ self.W
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        # Your code goes here. ################################################
        # self.gradW = ... ; self.gradb = ...
        self.gradW += gradOutput.T @ input
        self.gradb += gradOutput.sum(axis=0)
        pass

    def zeroGradParameters(self):
        self.gradW.fill(0)
        self.gradb.fill(0)

    def getParameters(self):
        return [self.W, self.b]

    def getGradParameters(self):
        return [self.gradW, self.gradb]

    def __repr__(self):
        s = self.W.shape
        q = 'Linear %d -> %d' %(s[1],s[0])
        return q

## 2. (0.2) SoftMax
- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

$\text{softmax}(x)_i = \frac{\exp x_i} {\sum_j \exp x_j}$

Recall that $\text{softmax}(x) == \text{softmax}(x - \text{const})$. It makes possible to avoid computing exp() from large argument.

In [5]:
class SoftMax(Module):
    def __init__(self):
         super(SoftMax, self).__init__()

    def updateOutput(self, input):
        self.output = np.subtract(input, input.max(axis=1, keepdims=True))
        exp_input = np.exp(self.output)
        self.output = exp_input / exp_input.sum(axis=1, keepdims=True)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = self.output * (
            gradOutput - np.sum(gradOutput * self.output, axis=1, keepdims=True)
        )
        return self.gradInput

    def __repr__(self):
        return "SoftMax"

## 3. (0.2) LogSoftMax
- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

$\text{logsoftmax}(x)_i = \log\text{softmax}(x)_i = x_i - \log {\sum_j \exp x_j}$

The main goal of this layer is to be used in computation of log-likelihood loss.

In [33]:
class LogSoftMax(Module):
    def __init__(self):
         super(LogSoftMax, self).__init__()

    def updateOutput(self, input):
        shifted = input - np.max(input, axis=1, keepdims=True)
        logsumexp = np.log(np.sum(np.exp(shifted), axis=1, keepdims=True))
        self.output = shifted - logsumexp
        return self.output

    def updateGradInput(self, input, gradOutput):
        softmax = np.exp(self.output)
        self.gradInput = gradOutput - softmax * np.sum(gradOutput, axis=1, keepdims=True)
        return self.gradInput

    def __repr__(self):
        return "LogSoftMax"

## 4. (0.3) Batch normalization
One of the most significant recent ideas that impacted NNs a lot is [**Batch normalization**](http://arxiv.org/abs/1502.03167). The idea is simple, yet effective: the features should be whitened ($mean = 0$, $std = 1$) all the way through NN. This improves the convergence for deep models letting it train them for days but not weeks. **You are** to implement the first part of the layer: features normalization. The second part (`ChannelwiseScaling` layer) is implemented below.

- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

The layer should work as follows. While training (`self.training == True`) it transforms input as $$y = \frac{x - \mu}  {\sqrt{\sigma + \epsilon}}$$
where $\mu$ and $\sigma$ - mean and variance of feature values in **batch** and $\epsilon$ is just a small number for numericall stability. Also during training, layer should maintain exponential moving average values for mean and variance:
```
    self.moving_mean = self.moving_mean * alpha + batch_mean * (1 - alpha)
    self.moving_variance = self.moving_variance * alpha + batch_variance * (1 - alpha)
```
During testing (`self.training == False`) the layer normalizes input using moving_mean and moving_variance.

Note that decomposition of batch normalization on normalization itself and channelwise scaling here is just a common **implementation** choice. In general "batch normalization" always assumes normalization + scaling.

In [ ]:
class BatchNormalization(Module):
    EPS = 1e-5

    def __init__(self, alpha=0.9):
        super(BatchNormalization, self).__init__()
        self.alpha = alpha

        self.moving_mean = None
        self.moving_variance = None

        self.batch_mean = None
        self.batch_variance = None
        self.centered_input = None
        self.std_inv = None

    def updateOutput(self, input):
        if self.moving_mean is None:
            self.moving_mean = np.zeros(input.shape[1], dtype=np.float32)
            self.moving_variance = np.ones(input.shape[1], dtype=np.float32)

        if self.training:
            x = torch.from_numpy(input.astype(np.float32))

            self.batch_mean = x.mean(dim=0).numpy()
            self.batch_variance = x.var(dim=0, unbiased=False).numpy()

            self.centered_input = input - self.batch_mean
            self.std_inv = 1.0 / np.sqrt(self.batch_variance + self.EPS)
            self.output = self.centered_input * self.std_inv

            # для running variance torch обновляет через unbiased variance
            batch_var_unbiased = x.var(dim=0, unbiased=True).numpy()

            self.moving_mean = (
                self.alpha * self.moving_mean + (1.0 - self.alpha) * self.batch_mean
            ).astype(np.float32)

            self.moving_variance = (
                self.alpha * self.moving_variance + (1.0 - self.alpha) * batch_var_unbiased
            ).astype(np.float32)
        else:
            self.output = (input - self.moving_mean) / np.sqrt(self.moving_variance + self.EPS)

        return self.output

    def updateGradInput(self, input, gradOutput):
        mean_grad = np.mean(gradOutput, axis=0)
        mean_grad_xhat = np.mean(gradOutput * self.output, axis=0)

        self.gradInput = (1.0 / np.sqrt(self.batch_variance + self.EPS)) * (
            gradOutput - mean_grad - self.output * mean_grad_xhat
        )

        return self.gradInput

    def __repr__(self):
        return "BatchNormalization"

In [ ]:
class ChannelwiseScaling(Module):
    def __init__(self, n_out):
        super(ChannelwiseScaling, self).__init__()

        self.gamma = np.ones(n_out, dtype=np.float32)
        self.beta = np.zeros(n_out, dtype=np.float32)

        self.gradGamma = np.zeros_like(self.gamma)
        self.gradBeta = np.zeros_like(self.beta)

    def updateOutput(self, input):
        self.output = input * self.gamma[None, :] + self.beta[None, :]
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput * self.gamma[None, :]
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        self.gradGamma = np.sum(input * gradOutput, axis=0)
        self.gradBeta = np.sum(gradOutput, axis=0)

    def zeroGradParameters(self):
        self.gradGamma.fill(0)
        self.gradBeta.fill(0)

    def getParameters(self):
        return [self.gamma, self.beta]

    def getGradParameters(self):
        return [self.gradGamma, self.gradBeta]

    def __repr__(self):
        return "ChannelwiseScaling"

Practical notes. If BatchNormalization is placed after a linear transformation layer (including dense layer, convolutions, channelwise scaling) that implements function like `y = weight * x + bias`, than bias adding become useless and could be omitted since its effect will be discarded while batch mean subtraction. If BatchNormalization (followed by `ChannelwiseScaling`) is placed before a layer that propagates scale (including ReLU, LeakyReLU) followed by any linear transformation layer than parameter `gamma` in `ChannelwiseScaling` could be freezed since it could be absorbed into the linear transformation layer.

## 5. (0.3) Dropout
Implement [**dropout**](https://www.cs.toronto.edu/~hinton/absps/JMLRdropout.pdf). The idea and implementation is really simple: just multimply the input by $Bernoulli(p)$ mask. Here $p$ is probability of an element to be zeroed.

This has proven to be an effective technique for regularization and preventing the co-adaptation of neurons.

While training (`self.training == True`) it should sample a mask on each iteration (for every batch), zero out elements and multiply elements by $1 / (1 - p)$. The latter is needed for keeping mean values of features close to mean values which will be in test mode. When testing this module should implement identity transform i.e. `self.output = input`.

- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

In [9]:
class Dropout(Module):
    def __init__(self, p=0.5):
        super(Dropout, self).__init__()

        self.p = p
        self.mask = None

    def updateOutput(self, input):
        # Your code goes here. ################################################
        if self.training:
            self.mask = (np.random.rand(*input.shape) > self.p).astype(input.dtype)
            self.output = input * self.mask / (1.0 - self.p)
        else:
            self.output = input
        return self.output

    def updateGradInput(self, input, gradOutput):
        # Your code goes here. ################################################
        if self.training:
            self.gradInput = gradOutput * self.mask / (1.0 - self.p)
        else:
            self.gradInput = gradOutput
        return self.gradInput

    def __repr__(self):
        return "Dropout"

#6. (2.0) Conv2d
Implement [**Conv2d**](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html). Use only this list of parameters: (in_channels, out_channels, kernel_size, stride, padding, bias, padding_mode) and fix dilation=1 and groups=1.

In [10]:
import numpy as np


class Conv2d(Module):
    def __init__(self, in_channels, out_channels, kernel_size,
                 stride=1, padding=0, bias=True, padding_mode='zeros'):
        super(Conv2d, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size if isinstance(kernel_size, tuple) else (kernel_size, kernel_size)
        self.stride = stride if isinstance(stride, tuple) else (stride, stride)
        self.padding = padding
        self.use_bias = bias
        self.padding_mode = padding_mode

        kh, kw = self.kernel_size
        stdv = 1.0 / np.sqrt(in_channels * kh * kw)

        self.weight = np.random.uniform(
            -stdv, stdv, size=(out_channels, in_channels, kh, kw)
        ).astype(np.float32)
        self.gradWeight = np.zeros_like(self.weight)

        if self.use_bias:
            self.bias = np.random.uniform(-stdv, stdv, size=(out_channels,)).astype(np.float32)
            self.gradBias = np.zeros_like(self.bias)
        else:
            self.bias = None
            self.gradBias = None

        # алиасы, если где-то в ноуте ожидаются старые имена
        self.W = self.weight
        self.gradW = self.gradWeight
        self.b = self.bias
        self.gradb = self.gradBias

    def _get_padding(self, input_shape):
        _, _, H, W = input_shape
        kh, kw = self.kernel_size
        sh, sw = self.stride

        if self.padding == 'same':
            out_h = int(np.ceil(H / sh))
            out_w = int(np.ceil(W / sw))

            pad_h_total = max((out_h - 1) * sh + kh - H, 0)
            pad_w_total = max((out_w - 1) * sw + kw - W, 0)

            ph_top = pad_h_total // 2
            ph_bottom = pad_h_total - ph_top
            pw_left = pad_w_total // 2
            pw_right = pad_w_total - pw_left

            return ph_top, ph_bottom, pw_left, pw_right

        if isinstance(self.padding, tuple):
            ph, pw = self.padding
        else:
            ph, pw = self.padding, self.padding

        return ph, ph, pw, pw

    def _pad_input(self, input):
        ph_top, ph_bottom, pw_left, pw_right = self._get_padding(input.shape)

        if ph_top == 0 and ph_bottom == 0 and pw_left == 0 and pw_right == 0:
            return input

        if self.padding_mode == 'zeros':
            return np.pad(
                input,
                ((0, 0), (0, 0), (ph_top, ph_bottom), (pw_left, pw_right)),
                mode='constant',
                constant_values=0
            )
        elif self.padding_mode == 'replicate':
            return np.pad(
                input,
                ((0, 0), (0, 0), (ph_top, ph_bottom), (pw_left, pw_right)),
                mode='edge'
            )
        elif self.padding_mode == 'reflect':
            return np.pad(
                input,
                ((0, 0), (0, 0), (ph_top, ph_bottom), (pw_left, pw_right)),
                mode='reflect'
            )
        else:
            raise ValueError(f"Unsupported padding_mode: {self.padding_mode}")

    def _map_reflect_index(self, idx, size):
        while idx < 0 or idx >= size:
            if idx < 0:
                idx = -idx
            if idx >= size:
                idx = 2 * size - 2 - idx
        return idx

    def _unpad_grad(self, grad_padded, input):
        ph_top, ph_bottom, pw_left, pw_right = self._get_padding(input.shape)
        H, W = input.shape[2], input.shape[3]

        if self.padding_mode == 'zeros':
            return grad_padded[:, :, ph_top:ph_top + H, pw_left:pw_left + W]

        grad_input = np.zeros_like(input)
        N, C, H_p, W_p = grad_padded.shape

        for n in range(N):
            for c in range(C):
                for i in range(H_p):
                    ii = i - ph_top
                    if self.padding_mode == 'replicate':
                        ii = min(max(ii, 0), H - 1)
                    elif self.padding_mode == 'reflect':
                        ii = self._map_reflect_index(ii, H)

                    for j in range(W_p):
                        jj = j - pw_left
                        if self.padding_mode == 'replicate':
                            jj = min(max(jj, 0), W - 1)
                        elif self.padding_mode == 'reflect':
                            jj = self._map_reflect_index(jj, W)

                        grad_input[n, c, ii, jj] += grad_padded[n, c, i, j]

        return grad_input

    def updateOutput(self, input):
        batch_size, in_channels, H, W = input.shape
        kh, kw = self.kernel_size
        sh, sw = self.stride

        x_padded = self._pad_input(input)
        H_p, W_p = x_padded.shape[2], x_padded.shape[3]

        H_out = (H_p - kh) // sh + 1
        W_out = (W_p - kw) // sw + 1

        self.output = np.zeros(
            (batch_size, self.out_channels, H_out, W_out),
            dtype=input.dtype
        )

        for n in range(batch_size):
            for oc in range(self.out_channels):
                for i in range(H_out):
                    for j in range(W_out):
                        h_start = i * sh
                        w_start = j * sw
                        window = x_padded[n, :, h_start:h_start + kh, w_start:w_start + kw]
                        self.output[n, oc, i, j] = np.sum(window * self.weight[oc])
                        if self.use_bias:
                            self.output[n, oc, i, j] += self.bias[oc]

        return self.output

    def updateGradInput(self, input, gradOutput):
        batch_size, in_channels, H, W = input.shape
        kh, kw = self.kernel_size
        sh, sw = self.stride

        x_padded = self._pad_input(input)
        gradInput_padded = np.zeros_like(x_padded)

        H_out, W_out = gradOutput.shape[2], gradOutput.shape[3]

        for n in range(batch_size):
            for oc in range(self.out_channels):
                for i in range(H_out):
                    for j in range(W_out):
                        h_start = i * sh
                        w_start = j * sw
                        gradInput_padded[n, :, h_start:h_start + kh, w_start:w_start + kw] += (
                            gradOutput[n, oc, i, j] * self.weight[oc]
                        )

        self.gradInput = self._unpad_grad(gradInput_padded, input)
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        kh, kw = self.kernel_size
        sh, sw = self.stride

        x_padded = self._pad_input(input)
        H_out, W_out = gradOutput.shape[2], gradOutput.shape[3]

        self.gradWeight.fill(0)
        if self.use_bias:
            self.gradBias.fill(0)

        for n in range(input.shape[0]):
            for oc in range(self.out_channels):
                for i in range(H_out):
                    for j in range(W_out):
                        h_start = i * sh
                        w_start = j * sw
                        window = x_padded[n, :, h_start:h_start + kh, w_start:w_start + kw]
                        self.gradWeight[oc] += gradOutput[n, oc, i, j] * window
                        if self.use_bias:
                            self.gradBias[oc] += gradOutput[n, oc, i, j]

        # синхронизируем алиасы
        self.gradW = self.gradWeight
        self.gradb = self.gradBias

    def zeroGradParameters(self):
        self.gradWeight.fill(0)
        if self.use_bias:
            self.gradBias.fill(0)

        self.gradW = self.gradWeight
        self.gradb = self.gradBias

    def getParameters(self):
        if self.use_bias:
            return [self.weight, self.bias]
        return [self.weight]

    def getGradParameters(self):
        if self.use_bias:
            return [self.gradWeight, self.gradBias]
        return [self.gradWeight]

    def __repr__(self):
        return (
            f"Conv2d(in_channels={self.in_channels}, "
            f"out_channels={self.out_channels}, "
            f"kernel_size={self.kernel_size}, "
            f"stride={self.stride}, "
            f"padding={self.padding}, "
            f"padding_mode={self.padding_mode})"
        )

#7. (0.5) Implement [**MaxPool2d**](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) and [**AvgPool2d**](https://pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html). Use only parameters like kernel_size, stride, padding (negative infinity for maxpool and zero for avgpool) and other parameters fixed as in framework.

In [11]:
class MaxPool2d(Module):
    def __init__(self, kernel_size, stride, padding):
        super(MaxPool2d, self).__init__()

        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

    def updateOutput(self, input):
        k_h, k_w = self.kernel_size if isinstance(self.kernel_size, tuple) else (self.kernel_size, self.kernel_size)
        s_h, s_w = self.stride if isinstance(self.stride, tuple) else (self.stride, self.stride)
        p_h, p_w = self.padding if isinstance(self.padding, tuple) else (self.padding, self.padding)

        batch_size, channels, in_h, in_w = input.shape

        padded_input = np.pad(
            input,
            ((0, 0), (0, 0), (p_h, p_h), (p_w, p_w)),
            mode='constant',
            constant_values=-np.inf
        )

        out_h = (in_h + 2 * p_h - k_h) // s_h + 1
        out_w = (in_w + 2 * p_w - k_w) // s_w + 1

        self.output = np.zeros((batch_size, channels, out_h, out_w))
        self.max_indices = np.zeros((batch_size, channels, out_h, out_w, 2), dtype=int)
        self.input_shape = input.shape

        for n in range(batch_size):
            for c in range(channels):
                for i in range(out_h):
                    for j in range(out_w):
                        h_start = i * s_h
                        h_end = h_start + k_h
                        w_start = j * s_w
                        w_end = w_start + k_w

                        window = padded_input[n, c, h_start:h_end, w_start:w_end]
                        self.output[n, c, i, j] = np.max(window)

                        max_pos = np.unravel_index(np.argmax(window), window.shape)
                        self.max_indices[n, c, i, j] = (h_start + max_pos[0], w_start + max_pos[1])

        return self.output

    def updateGradInput(self, input, gradOutput):
        batch_size, channels, in_h, in_w = self.input_shape
        p_h, p_w = self.padding if isinstance(self.padding, tuple) else (self.padding, self.padding)

        grad_padded = np.zeros((batch_size, channels, in_h + 2 * p_h, in_w + 2 * p_w))
        _, _, out_h, out_w = gradOutput.shape

        for n in range(batch_size):
            for c in range(channels):
                for i in range(out_h):
                    for j in range(out_w):
                        h_idx, w_idx = self.max_indices[n, c, i, j]
                        grad_padded[n, c, h_idx, w_idx] += gradOutput[n, c, i, j]

        if p_h > 0 or p_w > 0:
            self.gradInput = grad_padded[:, :, p_h:p_h + in_h, p_w:p_w + in_w]
        else:
            self.gradInput = grad_padded

        return self.gradInput

    def __repr__(self):
        return "MaxPool2d"


class AvgPool2d(Module):
    def __init__(self, kernel_size, stride, padding):
        super(AvgPool2d, self).__init__()

        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

    def updateOutput(self, input):
        k_h, k_w = self.kernel_size if isinstance(self.kernel_size, tuple) else (self.kernel_size, self.kernel_size)
        s_h, s_w = self.stride if isinstance(self.stride, tuple) else (self.stride, self.stride)
        p_h, p_w = self.padding if isinstance(self.padding, tuple) else (self.padding, self.padding)

        batch_size, channels, in_h, in_w = input.shape

        padded_input = np.pad(
            input,
            ((0, 0), (0, 0), (p_h, p_h), (p_w, p_w)),
            mode='constant',
            constant_values=0
        )

        out_h = (in_h + 2 * p_h - k_h) // s_h + 1
        out_w = (in_w + 2 * p_w - k_w) // s_w + 1

        self.output = np.zeros((batch_size, channels, out_h, out_w))
        self.input_shape = input.shape

        for n in range(batch_size):
            for c in range(channels):
                for i in range(out_h):
                    for j in range(out_w):
                        h_start = i * s_h
                        h_end = h_start + k_h
                        w_start = j * s_w
                        w_end = w_start + k_w

                        window = padded_input[n, c, h_start:h_end, w_start:w_end]
                        self.output[n, c, i, j] = np.mean(window)

        return self.output

    def updateGradInput(self, input, gradOutput):
        k_h, k_w = self.kernel_size if isinstance(self.kernel_size, tuple) else (self.kernel_size, self.kernel_size)
        s_h, s_w = self.stride if isinstance(self.stride, tuple) else (self.stride, self.stride)
        p_h, p_w = self.padding if isinstance(self.padding, tuple) else (self.padding, self.padding)

        batch_size, channels, in_h, in_w = self.input_shape
        _, _, out_h, out_w = gradOutput.shape

        grad_padded = np.zeros((batch_size, channels, in_h + 2 * p_h, in_w + 2 * p_w))
        coeff = 1.0 / (k_h * k_w)

        for n in range(batch_size):
            for c in range(channels):
                for i in range(out_h):
                    for j in range(out_w):
                        h_start = i * s_h
                        h_end = h_start + k_h
                        w_start = j * s_w
                        w_end = w_start + k_w

                        grad_padded[n, c, h_start:h_end, w_start:w_end] += gradOutput[n, c, i, j] * coeff

        if p_h > 0 or p_w > 0:
            self.gradInput = grad_padded[:, :, p_h:p_h + in_h, p_w:p_w + in_w]
        else:
            self.gradInput = grad_padded

        return self.gradInput

    def __repr__(self):
        return "AvgPool2d"

#8. (0.3) Implement **GlobalMaxPool2d** and **GlobalAvgPool2d**. They do not have testing and parameters are up to you but they must aggregate information within channels. Write test functions for these layers on your own.

In [12]:
class GlobalMaxPool2d(Module):
    def __init__(self):
        super(GlobalMaxPool2d, self).__init__()

    def updateOutput(self, input):
        # input: (N, C, H, W)
        self.input_shape = input.shape
        self.output = np.max(input, axis=(2, 3))

        # запоминаем позиции максимумов для backward
        n, c, h, w = input.shape
        flat_input = input.reshape(n, c, -1)
        self.max_indices = np.argmax(flat_input, axis=2)  # shape: (N, C)

        return self.output

    def updateGradInput(self, input, gradOutput):
        # gradOutput: (N, C)
        n, c, h, w = input.shape

        self.gradInput = np.zeros_like(input)
        flat_grad = self.gradInput.reshape(n, c, -1)

        for i in range(n):
            for j in range(c):
                flat_grad[i, j, self.max_indices[i, j]] = gradOutput[i, j]

        self.gradInput = flat_grad.reshape(n, c, h, w)
        return self.gradInput

    def __repr__(self):
        return "GlobalMaxPool2d"


class GlobalAvgPool2d(Module):
    def __init__(self):
        super(GlobalAvgPool2d, self).__init__()

    def updateOutput(self, input):
        # input: (N, C, H, W)
        self.input_shape = input.shape
        self.output = np.mean(input, axis=(2, 3))
        return self.output

    def updateGradInput(self, input, gradOutput):
        # gradOutput: (N, C)
        n, c, h, w = input.shape

        self.gradInput = np.zeros_like(input)
        coeff = 1.0 / (h * w)

        for i in range(n):
            for j in range(c):
                self.gradInput[i, j, :, :] = gradOutput[i, j] * coeff

        return self.gradInput

    def __repr__(self):
        return "GlobalAvgPool2d"

#9. (0.2) Implement [**Flatten**](https://pytorch.org/docs/stable/generated/torch.flatten.html)

In [ ]:
class Flatten(Module):
    def __init__(self, start_dim=1, end_dim=-1):
        super(Flatten, self).__init__()
        self.start_dim = start_dim
        self.end_dim = end_dim
        self.input_shape = None

    def updateOutput(self, input):
        self.input_shape = input.shape
        ndim = input.ndim

        start_dim = self.start_dim if self.start_dim >= 0 else self.start_dim + ndim
        end_dim = self.end_dim if self.end_dim >= 0 else self.end_dim + ndim

        if start_dim > end_dim:
            raise ValueError("start_dim must be <= end_dim")

        flattened_dim = int(np.prod(input.shape[start_dim:end_dim + 1]))
        new_shape = (
            list(input.shape[:start_dim]) +
            [flattened_dim] +
            list(input.shape[end_dim + 1:])
        )

        self.output = input.reshape(*new_shape)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput.reshape(self.input_shape)
        return self.gradInput

    def __repr__(self):
        return f"Flatten(start_dim={self.start_dim}, end_dim={self.end_dim})"

# Activation functions

Here's the complete example for the **Rectified Linear Unit** non-linearity (aka **ReLU**):

In [14]:
class ReLU(Module):
    def __init__(self):
         super(ReLU, self).__init__()

    def updateOutput(self, input):
        self.output = np.maximum(input, 0)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = np.multiply(gradOutput , input > 0)
        return self.gradInput

    def __repr__(self):
        return "ReLU"

## 10. (0.1) Leaky ReLU
Implement [**Leaky Rectified Linear Unit**](http://en.wikipedia.org/wiki%2FRectifier_%28neural_networks%29%23Leaky_ReLUs). Expriment with slope.

In [15]:
class LeakyReLU(Module):
    def __init__(self, slope = 0.03):
        super(LeakyReLU, self).__init__()

        self.slope = slope

    def updateOutput(self, input):
        self.output = np.where(input > 0, input, self.slope * input)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput * np.where(input > 0, 1.0, self.slope)
        return self.gradInput

    def __repr__(self):
        return "LeakyReLU"

## 11. (0.1) ELU
Implement [**Exponential Linear Units**](http://arxiv.org/abs/1511.07289) activations.

In [16]:
class ELU(Module):
    def __init__(self, alpha = 1.0):
        super(ELU, self).__init__()

        self.alpha = alpha

    def updateOutput(self, input):
        self.output = np.where(input > 0, input, self.alpha * (np.exp(input) - 1))
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput * np.where(input > 0, 1.0, self.alpha * np.exp(input))
        return self.gradInput

    def __repr__(self):
        return "ELU"

## 12. (0.1) SoftPlus
Implement [**SoftPlus**](https://en.wikipedia.org/wiki%2FRectifier_%28neural_networks%29) activations. Look, how they look a lot like ReLU.

In [17]:
class SoftPlus(Module):
    def __init__(self):
        super(SoftPlus, self).__init__()

    def updateOutput(self, input):
        self.output = np.log1p(np.exp(-np.abs(input))) + np.maximum(input, 0)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput * (1.0 / (1.0 + np.exp(-input)))
        return self.gradInput

    def __repr__(self):
        return "SoftPlus"

#13. (0.2) Gelu
Implement [**Gelu**](https://pytorch.org/docs/stable/generated/torch.nn.GELU.html) activations.

In [18]:
import numpy as np
import math

class Gelu(Module):
    def __init__(self):
        super(Gelu, self).__init__()

    def updateOutput(self, input):
        erf_vec = np.vectorize(math.erf)
        self.output = 0.5 * input * (1.0 + erf_vec(input / np.sqrt(2.0)))
        return self.output

    def updateGradInput(self, input, gradOutput):
        erf_vec = np.vectorize(math.erf)
        deriv = 0.5 * (1.0 + erf_vec(input / np.sqrt(2.0))) + \
                input * np.exp(-input ** 2 / 2.0) / np.sqrt(2.0 * np.pi)
        self.gradInput = gradOutput * deriv
        return self.gradInput

    def __repr__(self):
        return "Gelu"

# Criterions

Criterions are used to score the models answers.

In [19]:
class Criterion(object):
    def __init__ (self):
        self.output = None
        self.gradInput = None

    def forward(self, input, target):
        """
            Given an input and a target, compute the loss function
            associated to the criterion and return the result.

            For consistency this function should not be overrided,
            all the code goes in `updateOutput`.
        """
        return self.updateOutput(input, target)

    def backward(self, input, target):
        """
            Given an input and a target, compute the gradients of the loss function
            associated to the criterion and return the result.

            For consistency this function should not be overrided,
            all the code goes in `updateGradInput`.
        """
        return self.updateGradInput(input, target)

    def updateOutput(self, input, target):
        """
        Function to override.
        """
        return self.output

    def updateGradInput(self, input, target):
        """
        Function to override.
        """
        return self.gradInput

    def __repr__(self):
        """
        Pretty printing. Should be overrided in every module if you want
        to have readable description.
        """
        return "Criterion"

The **MSECriterion**, which is basic L2 norm usually used for regression, is implemented here for you.
- input:   **`batch_size x n_feats`**
- target: **`batch_size x n_feats`**
- output: **scalar**

In [20]:
class MSECriterion(Criterion):
    def __init__(self):
        super(MSECriterion, self).__init__()

    def updateOutput(self, input, target):
        self.output = np.sum(np.power(input - target, 2)) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        self.gradInput  = (input - target) * 2 / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "MSECriterion"

## 14. (0.2) Negative LogLikelihood criterion (numerically unstable)
You task is to implement the **ClassNLLCriterion**. It should implement [multiclass log loss](http://scikit-learn.org/stable/modules/model_evaluation.html#log-loss). Nevertheless there is a sum over `y` (target) in that formula,
remember that targets are one-hot encoded. This fact simplifies the computations a lot. Note, that criterions are the only places, where you divide by batch size. Also there is a small hack with adding small number to probabilities to avoid computing log(0).
- input:   **`batch_size x n_feats`** - probabilities
- target: **`batch_size x n_feats`** - one-hot representation of ground truth
- output: **scalar**



In [21]:
class ClassNLLCriterionUnstable(Criterion):
    EPS = 1e-15
    def __init__(self):
        a = super(ClassNLLCriterionUnstable, self)
        super(ClassNLLCriterionUnstable, self).__init__()

    def updateOutput(self, input, target):

        # Use this trick to avoid numerical errors
        input_clamp = np.clip(input, self.EPS, 1 - self.EPS)

        self.output = -np.sum(target * np.log(input_clamp)) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):

        # Use this trick to avoid numerical errors
        input_clamp = np.clip(input, self.EPS, 1 - self.EPS)

        self.gradInput = -target / input_clamp / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "ClassNLLCriterionUnstable"

## 15. (0.3) Negative LogLikelihood criterion (numerically stable)
- input:   **`batch_size x n_feats`** - log probabilities
- target: **`batch_size x n_feats`** - one-hot representation of ground truth
- output: **scalar**

Task is similar to the previous one, but now the criterion input is the output of log-softmax layer. This decomposition allows us to avoid problems with computation of forward and backward of log().

In [22]:
class ClassNLLCriterion(Criterion):
    def __init__(self):
        a = super(ClassNLLCriterion, self)
        super(ClassNLLCriterion, self).__init__()

    def updateOutput(self, input, target):
        self.output = -np.sum(target * input) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        self.gradInput = -target / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "ClassNLLCriterion"

1-я часть задания: реализация слоев, лосей и функций активации - 5 баллов. \\
2-я часть задания: реализация моделей на своих классах. Что должно быть:
  1. Выберите оптимизатор и реализуйте его, чтоб он работал с вами классами. - 1 балл.
  2. Модель для задачи мультирегрессии на выбраных вами данных. Использовать FCNN, dropout, batchnorm, MSE. Пробуйте различные фукнции активации. Для первой модели попробуйте большую, среднюю и маленькую модель. - 1 балл.
  3. Модель для задачи мультиклассификации на MNIST. Использовать свёртки, макспулы, флэттэны, софтмаксы - 1 балла.
  4. Автоэнкодер для выбранных вами данных. Должен быть на свёртках и полносвязных слоях, дропаутах, батчнормах и тд. - 2 балла. \\

Дополнительно в оценке каждой модели будет учитываться:
1. Наличие правильно выбранной метрики и лосс функции.
2. Отрисовка графиков лосей и метрик на трейне-валидации. Проверка качества модели на тесте.
3. Наличие шедулера для lr.
4. Наличие вормапа.
5. Наличие механизма ранней остановки и сохранение лучшей модели.
6. Свитч лося (метрики) и оптимайзера.